# Pobieranie i scalanie danych reanalizy ERA5-Land (Kraków)

## Konfiguracja i importowanie bibliotek

In [ ]:
import os
import glob
from pathlib import Path 
import sys
import xarray as xr 
import cdsapi

# Dodanie katalogu głównego projektu do sys.path
root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

# Utworzenie katalogów na dane
from src.config import ERA5_DATA_DIR, ERA5_NETCDF_PATH

ERA5_DATA_DIR.mkdir(parents=True, exist_ok=True)
ERA5_NETCDF_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Katalog dla danych ERA5 został przygotowany.")

## Pobieranie danych

In [ ]:
dataset = "reanalysis-era5-land"
client = cdsapi.Client()

# c = cdsapi.Client(url="https://cds.climate.copernicus.eu/api",
#                         key="f85a23bc-7b23-4cea-92fa-ec62fa5810fc")

krakow_area = [50.2, 19.7, 49.9, 20.2]

for year in range(2000, 2026):
    for month in range(1, 13):
        
        filename = ERA5_DATA_DIR / f"krakow_temp_{year}_{month:02d}.nc"
        
        if filename.exists():
            print(f"Pominięto: {filename}. Plik już istnieje.")
            continue

        print(f"Wysyłam zapytanie: {year}-{month:02d}")

        request = {
            "variable": ["2m_temperature"],
            "year": str(year),
            "month": f"{month:02d}",
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": [f"{h:02d}:00" for h in range(24)],
            "data_format": "netcdf",
            "download_format": "unarchived", 
            "area": krakow_area,
        }

        try:
            c.retrieve(dataset, request).download(filename)
            print(f"Pobrano pomyślnie: {filename}")
        except Exception as e:
            print(f"Błąd dla {year}-{month:02d}: {e}")
            if year >= 2025:
                print("Przerwanie: dotarliśmy do końca dostępnych danych.")
                break
print("Pobieranie zakończone.")

## Scalanie plików w jeden zbiór

In [ ]:
# Pobranie listy wszystkich pobranych plików rocznych posortowanych chronologicznie
nc_files = sorted(glob.glob(str(ERA5_DATA_DIR / "krakow_temp_*.nc")))
if not nc_files:
    print("Brak plików do scalenia!")
    return
print(f"Znaleziono {len(nc_files)} plików do scalenia.")

# Scalenie wzdłuż wymiaru czasu za pomocą xarray
ds = xr.open_mfdataset(nc_files, combine='by_coords', chunks={'valid_time': 1000})

# Zapis scalonego zestawu danych do jednego pliku docelowego
ds.to_netcdf(ERA5_NETCDF_PATH)
print(f"Pomyślnie zapisano scalony plik: {ERA5_NETCDF_PATH}")

## Wczytanie scalonego zbioru i weryfikacja danych

In [5]:
era5_file = xr.open_dataset(ERA5_NETCDF_PATH)
print(era5_file)

<xarray.Dataset> Size: 27MB
Dimensions:     (valid_time: 227928, latitude: 4, longitude: 6)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 2MB 2000-01-01 ... 2025-12-31T23:...
    expver      (valid_time) <U4 4MB ...
  * latitude    (latitude) float64 32B 50.2 50.1 50.0 49.9
  * longitude   (longitude) float64 48B 19.7 19.8 19.9 20.0 20.1 20.2
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 22MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-11T14:34 GRIB to CDM+CF via cfgrib-0.9.1...
